In [1]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages,MessagesState
from dotenv import load_dotenv 
from langchain_groq import ChatGroq
from IPython.display import Image,display
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
import os 


In [2]:
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGSMITH_PROJECT")

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="qwen/qwen3.6-27b")


In [4]:
@tool 
def getweather(city:str)->str:
    """get the weather of the city will i will provide to you """
    return f"the weather of the {city} is 78 f"

@tool 
def pincode_db_q(qurey:str)->str:

    """your db master and the expert now you need to arrange the get write the quries """
    return  qurey
tool=[getweather,pincode_db_q]



In [5]:
T_llm=llm.bind_tools(tool)

In [6]:
def bot_node(state:MessagesState):
    response=llm.invoke(state["messages"])

    return {"message":[response]}

In [7]:
tool_node = ToolNode(tool)

In [9]:
workflow=StateGraph(MessagesState)
workflow.add_node("botnode",bot_node)
workflow.add_node("tool",tool_node)

In [13]:
workflow.add_edge(START,bot_node)

workflow.add_conditional_edges(
    "botnode",
    tools_condition,
)

workflow.add_edge("tool", "agent")

app = workflow.compile()


ValueError: Branch with name `tools_condition` already exists for node `botnode`